# Import the library

In [1]:
import os

import torch
from trainer import Trainer, TrainerArgs

# from TTS.bin.compute_embeddings import compute_embeddings
from compute_embeddings import compute_embeddings # use custom formatter without forking the lib
from TTS.bin.resample import resample_files
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import CharactersConfig, Vits, VitsArgs, VitsAudioConfig

from TTS.tts.utils.languages import LanguageManager
from TTS.tts.utils.speakers import SpeakerManager
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

# from TTS.utils.downloaders import download_vctk
# from TTS.config import load_config
# from TTS.config.shared_configs import BaseDatasetConfig
# from TTS.tts.datasets import load_tts_samples
# from TTS.tts.utils.managers import save_file
# from TTS.tts.utils.speakers import SpeakerManager
from TTS.tts.datasets.formatters import vctk
import TTS.tts.datasets.formatters as formatters_module 
from functools import partial

from tqdm import tqdm

torch.set_num_threads(24)

# Setup constants

In [ ]:
# Current path
CURRENT_PATH = os.getcwd()

# Name of the run for the Trainer
RUN_NAME = "KhongKhunTTS-TH_EN-MultiSpeaker"

# Path where you want to save the models outputs (configs, checkpoints and tensorboard logs)
OUT_PATH = os.path.join(CURRENT_PATH, "runs")

# If you want to do transfer learning and speedup your training you can set here the path to the model
RESTORE_PATH = "./best_model_KhonkhunTTS_Tsync2-LJSpeech.pth"

# This paramter is useful to debug, it skips the training epochs and just do the evaluation  and produce the test sentences
SKIP_TRAIN_EPOCH = False

# Set here the batch size to be used in training and evaluation
BATCH_SIZE = 32

# Training Sampling rate and the target sampling rate for resampling the downloaded dataset (Note: If you change this you might need to redownload the dataset !!)
# Note: If you add new datasets, please make sure that the dataset sampling rate and this parameter are matching, otherwise resample your audios
SAMPLE_RATE = 16000

# Max audio length in seconds to be used in training (every audio bigger than it will be ignored)
MAX_AUDIO_LEN_IN_SECONDS = 10

# Define the number of threads used during the audio resampling
NUM_RESAMPLE_THREADS = 10

# Dataset configuration

In [3]:
import sys

original_vctk = vctk

def vctk_16k(root_path, meta_files=None, ignored_speakers=None):
    return original_vctk(
        root_path, 
        meta_files=meta_files, 
        wavs_path="wav16_silence_trimmed",
        mic="mic1",
        ignored_speakers=ignored_speakers
    )

setattr(formatters_module, 'vctk', vctk_16k)

tts_formatters = sys.modules['TTS.tts.datasets.formatters']
setattr(tts_formatters, 'vctk', vctk_16k)

tts_datasets = sys.modules['TTS.tts.datasets']
if hasattr(tts_datasets, 'vctk'):
    setattr(tts_datasets, 'vctk', vctk_16k)

import TTS.tts.datasets as datasets
if hasattr(datasets, 'vctk'):
    setattr(datasets, 'vctk', vctk_16k)

In [4]:
# init configs
commonvoice_config = BaseDatasetConfig(
    formatter="vctk",
    dataset_name="commonvoice",
    meta_file_train="",
    meta_file_val="",
    path=os.path.join(CURRENT_PATH, "commonvoice-to-vctk"),
    language="th",
    ignored_speakers=[
        "cv017", # Female Teenager
        "cv048", # Female Teenager
        "cv039", # Female Adult
        "cv052", # Female Adult
        "cv069", # Male Teenager
        "cv054", # Male Teenager
        "cv049", # Male Adult
        "cv026", # Male Adult
    ],
)

vctk_config = BaseDatasetConfig(
    formatter="vctk",
    dataset_name="vctk",
    meta_file_train="",
    meta_file_val="",
    path=os.path.join(CURRENT_PATH, "vctk-to-vctk"),
    language="en",
    ignored_speakers=[
        "p339", # F,American
        "p237", # M,Scottish
        "p249", # F,Scottish
        "p226", # M,English
        "p313", # F,Irish
        "p363", # M,Canadian
    ],
)

# Add here all datasets configs, in our case we just want to train with the VCTK dataset then we need to add just VCTK. Note: If you want to add new datasets, just add them here and it will automatically compute the speaker embeddings (d-vectors) for this new dataset :)
DATASETS_CONFIG_LIST = [commonvoice_config, vctk_config]

# Extract speaker embeddings

In [5]:
SPEAKER_ENCODER_CHECKPOINT_PATH = (
    "https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/model_se.pth.tar"
)
SPEAKER_ENCODER_CONFIG_PATH = "https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/config_se.json"

D_VECTOR_FILES = []  # List of speaker embeddings/d-vectors to be used during the training

# Iterates all the dataset configs checking if the speakers embeddings are already computated, if not compute it
for dataset_conf in DATASETS_CONFIG_LIST:
    # Check if the embeddings weren't already computed, if not compute it
    embeddings_file = os.path.join(dataset_conf.path, "dvector.pth")
    if not os.path.isfile(embeddings_file):
        print(f">>> Computing the speaker embeddings for the {dataset_conf.dataset_name} dataset")
        compute_embeddings(
            SPEAKER_ENCODER_CHECKPOINT_PATH,
            SPEAKER_ENCODER_CONFIG_PATH,
            embeddings_file,
            formatter_name=dataset_conf.formatter,
            formatter=vctk_16k if dataset_conf.formatter == "vctk_16k" else None,
            dataset_name=dataset_conf.dataset_name,
            dataset_path=dataset_conf.path,
            meta_file_train=dataset_conf.meta_file_train,
            meta_file_val=dataset_conf.meta_file_val,
        )

    D_VECTOR_FILES.append(embeddings_file)

>>> Computing the speaker embeddings for the commonvoice dataset
 | > Found 92956 files in /home/pruuwu/dubbing-ai/KhongKhunTTS/commonvoice-to-vctk
 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512
 | > power:1.5
 | > preemphasis:0.97
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:False
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:False
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:True
 | > db_level:-27.0
 | > stats_path:None
 | > base:10
 | > hop_length:160
 | > win_length:400


100%|██████████| 92956/92956 [13:07<00:00, 117.99it/s]


Speaker embeddings saved at: /home/pruuwu/dubbing-ai/KhongKhunTTS/commonvoice-to-vctk/dvector.pth
>>> Computing the speaker embeddings for the vctk dataset
 | > Found 44070 files in /home/pruuwu/dubbing-ai/KhongKhunTTS/vctk-to-vctk
 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512
 | > power:1.5
 | > preemphasis:0.97
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:False
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:False
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:True
 | > db_level:-27.0
 | > stats_path:None
 | > base:10
 | > hop_length:160
 | > win_length

100%|██████████| 44070/44070 [05:41<00:00, 128.87it/s]


Speaker embeddings saved at: /home/pruuwu/dubbing-ai/KhongKhunTTS/vctk-to-vctk/dvector.pth


# Audio config used in training.

In [6]:
audio_config = VitsAudioConfig(
    sample_rate=SAMPLE_RATE,
    hop_length=256,
    win_length=1024,
    fft_size=1024,
    mel_fmin=0.0,
    mel_fmax=None,
    num_mels=80,
)

# Model configuration

In [13]:
# Init VITSArgs setting the arguments that are needed for the KhongKhunTTS model
model_args = VitsArgs(
    d_vector_file=D_VECTOR_FILES,
    use_d_vector_file=True,
    d_vector_dim=512,
    num_layers_text_encoder=10,
    speaker_encoder_model_path=SPEAKER_ENCODER_CHECKPOINT_PATH,
    speaker_encoder_config_path=SPEAKER_ENCODER_CONFIG_PATH,
    resblock_type_decoder="2",  # In the YourTTS paper, trained using ResNet blocks type 2, if you like you can use the ResNet blocks type 1 like the VITS model
    # Useful parameters to enable the Speaker Consistency Loss (SCL) described in the paper
    use_speaker_encoder_as_loss=True,
    # Useful parameters to enable multilingual training
    use_language_embedding=True,
    embedded_language_dim=4,
)

In [ ]:
# General training config, here you can change the batch size and others useful parameters
config = VitsConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name=RUN_NAME,
    project_name="KhongKhunTTS",
    run_description="""
            - KhongKhunTTS trained using Commonvoice (VCTK structure) and VCTK by transfer learning from KhongkhunTTS TSync2 and LJSpeech
        """,
    dashboard_logger="tensorboard",
    logger_uri=None,
    audio=audio_config,
    batch_size=BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=BATCH_SIZE,
    num_loader_workers=8,
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=1000,
    save_step=5000,
    save_n_checkpoints=2,
    save_checkpoints=True,
    target_loss="loss_1",
    print_eval=False,
    use_phonemes=False,
    phonemizer="espeak",
    phoneme_language="en",
    compute_input_seq_cache=True,
    add_blank=True,
    text_cleaner="multilingual_cleaners",
    characters=CharactersConfig(
        characters_class="TTS.tts.models.vits.VitsCharacters",
        pad="_",
        eos="&",
        bos="*",
        blank=None,
        characters="ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz\u00af\u00b7\u00df\u00e0\u00e1\u00e2\u00e3\u00e4\u00e6\u00e7\u00e8\u00e9\u00ea\u00eb\u00ec\u00ed\u00ee\u00ef\u00f1\u00f2\u00f3\u00f4\u00f5\u00f6\u00f9\u00fa\u00fb\u00fc\u00ff\u0101\u0105\u0107\u0113\u0119\u011b\u012b\u0131\u0142\u0144\u014d\u0151\u0153\u015b\u016b\u0171\u017a\u017c\u01ce\u01d0\u01d2\u01d4\u0430\u0431\u0432\u0433\u0434\u0435\u0436\u0437\u0438\u0439\u043a\u043b\u043c\u043d\u043e\u043f\u0440\u0441\u0442\u0443\u0444\u0445\u0446\u0447\u0448\u0449\u044a\u044b\u044c\u044d\u044e\u044f\u0451\u0454\u0456\u0457\u0491\u2013!\"'(),-.:;?|~ \u0e01\u0e02\u0e04\u0e06\u0e07\u0e08\u0e09\u0e0a\u0e0b\u0e0c\u0e0d\u0e0e\u0e0f\u0e10\u0e11\u0e12\u0e13\u0e14\u0e15\u0e16\u0e17\u0e18\u0e19\u0e1a\u0e1b\u0e1c\u0e1d\u0e1e\u0e1f\u0e20\u0e21\u0e22\u0e23\u0e24\u0e25\u0e27\u0e28\u0e29\u0e2a\u0e2b\u0e2c\u0e2d\u0e2e\u0e2f\u0e30\u0e31\u0e32\u0e33\u0e34\u0e35\u0e36\u0e37\u0e38\u0e39\u0e40\u0e41\u0e42\u0e43\u0e44\u0e45\u0e46\u0e47\u0e48\u0e49\u0e4a\u0e4b\u0e4c\u0e4d\u2014\u2018\u2019\u201c\u201d",
        punctuations="!\"'(),-.:;?|~ ",
        phonemes="",
        is_unique=True,
        is_sorted=True,
    ),
    phoneme_cache_path=None,
    precompute_num_workers=12,
    start_by_longest=True,
    datasets=DATASETS_CONFIG_LIST,
    cudnn_benchmark=False,
    max_audio_len=SAMPLE_RATE * MAX_AUDIO_LEN_IN_SECONDS,
    mixed_precision=False,
    test_sentences=[
        # Thai - commonvoice
        [
            "ทดสอบการอ่านออกเสียงภาษาไทย",
            "VCTK_cv005",
            None,
            "th",
        ],
        [
            "ยักษ์ใหญ่ไล่ยักษ์เล็ก ยักษ์เล็กไล่ยักษ์ใหญ่",
            "VCTK_cv068",
            None,
            "th",
        ],
        [
            "ยายกินลำไย น้ำลายยายไหลย้อย",
            "VCTK_cv057",
            None,
            "th",
        ],
        [
            "ชามเขียวคว่ำเช้า ชามขาวคว่ำค่ำ",
            "VCTK_cv103",
            None,
            "th",
        ],
        [
            "หมอนลอยน้ำมา ว่ายน้ำไป ถอยหมอน",
            "VCTK_cv133",
            None,
            "th",
        ],
        [
            "เช้าฟาดผัดฟัก เย็นฟาดฟักผัด",
            "VCTK_cv128",
            None,
            "th",
        ],

        # English - VCTK
        [
            "English pronunciation test",
            "VCTK_p300", # F,American
            None,
            "en",
        ],
        [
            "She sells seashells by the seashore.",
            "VCTK_p271", # M,Scottish
            None,
            "en",
        ],
        [
            "A big black bug bit a big black dog on his big black nose.",
            "VCTK_p262", # F,Scottish
            None,
            "en",
        ],
        [
            "The quick brown fox jumps over the lazy dog.",
            "VCTK_p287", # M,English
            None,
            "en",
        ],
        [
            "I saw a kitten eating chicken in the kitchen.",
            "VCTK_p266", # F,Irish
            None,
            "en",
        ],
        [
            "I scream, you scream, we all scream for ice cream.",
            "VCTK_p302", # M,Canadian
            None,
            "en",
        ]
    ],
    # # Enable the weighted sampler
    use_weighted_sampler=True,
    # # Ensures that all speakers are seen in the training batch equally no matter how many samples each speaker has
    weighted_sampler_attrs={"speaker_name": 1.0},
    # weighted_sampler_multipliers={},
    weighted_sampler_multipliers={"Makeitnotblank": None},

    # It defines the Speaker Consistency Loss (SCL) α to 9 like the paper
    speaker_encoder_loss_alpha=9.0,
)

# Training

In [15]:
# Load all the datasets samples and split traning and evaluation sets
train_samples, eval_samples = load_tts_samples(
    config.datasets,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size
)

 | > Found 91809 files in /home/pruuwu/dubbing-ai/KhongKhunTTS/commonvoice-to-vctk
 | > Found 41777 files in /home/pruuwu/dubbing-ai/KhongKhunTTS/vctk-to-vctk


In [16]:
# Init the model
model = Vits.init_from_config(config)

 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512


In [17]:
# Init the trainer and 🚀
trainer = Trainer(
    TrainerArgs(restore_path=RESTORE_PATH, skip_train_epoch=SKIP_TRAIN_EPOCH),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 16
 | > Num. of Torch Threads: 24
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/home/pruuwu/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-TH_EN-MultiSpeaker-April-29-2025_03+05PM-8683cf7
 > Restoring from best_model_KhonkhunTTS_Tsync2-LJSpeech.pth ...


 > `speakers.pth` is saved to /home/pruuwu/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-TH_EN-MultiSpeaker-April-29-2025_03+05PM-8683cf7/speakers.pth.
 > `speakers_file` is updated in the config.json.
 > `language_ids.json` is saved to /home/pruuwu/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-TH_EN-MultiSpeaker-April-29-2025_03+05PM-8683cf7/language_ids.json.
 > `language_ids_file` is updated in the config.json.


 > Restoring Model...
 > Restoring Optimizer...
 > Partial model initialization...
 | > 899 / 899 layers are restored.
 > Model restored from step 76826
/home/pruuwu/.cache/pypoetry/virtualenvs/khongkhuntts-t5CDf9ik-py3.11/lib/python3.11/site-packages/trainer/trainer.py:561: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()

 > Model has 86142884 parameters


In [ ]:
trainer.fit()


 > EPOCH: 0/1000
 --> /home/pruuwu/dubbing-ai/KhongKhunTTS/runs/KhongKhunTTS-TH_EN-MultiSpeaker-April-29-2025_03+05PM-8683cf7




> DataLoader initialization
| > Tokenizer:
	| > add_blank: True
	| > use_eos_bos: False
	| > use_phonemes: False
| > Number of instances : 133330
 | > Preprocessing samples
 | > Max text length: 185
 | > Min text length: 2
 | > Avg text length: 33.632878521787426
 | 
 | > Max audio length: 159966.0
 | > Min audio length: 8686.5
 | > Avg audio length: 42818.68064499182
 | > Num. instances discarded samples: 88
 | > Batch group size: 1536.
 > Using weighted sampler for attribute 'speaker_name' with alpha '1.0'
None



 > TRAINING (2025-04-29 15:05:22) 


 > Attribute weights for '['VCTK_cv001', 'VCTK_cv002', 'VCTK_cv003', 'VCTK_cv004', 'VCTK_cv005', 'VCTK_cv006', 'VCTK_cv007', 'VCTK_cv008', 'VCTK_cv009', 'VCTK_cv010', 'VCTK_cv011', 'VCTK_cv012', 'VCTK_cv013', 'VCTK_cv014', 'VCTK_cv015', 'VCTK_cv016', 'VCTK_cv018', 'VCTK_cv019', 'VCTK_cv020', 'VCTK_cv021', 'VCTK_cv022', 'VCTK_cv023', 'VCTK_cv024', 'VCTK_cv025', 'VCTK_cv027', 'VCTK_cv028', 'VCTK_cv029', 'VCTK_cv030', 'VCTK_cv031', 'VCTK_cv032', 'VCTK_cv033', 'VCTK_cv034', 'VCTK_cv035', 'VCTK_cv036', 'VCTK_cv037', 'VCTK_cv038', 'VCTK_cv040', 'VCTK_cv041', 'VCTK_cv042', 'VCTK_cv043', 'VCTK_cv044', 'VCTK_cv045', 'VCTK_cv046', 'VCTK_cv047', 'VCTK_cv050', 'VCTK_cv051', 'VCTK_cv053', 'VCTK_cv055', 'VCTK_cv056', 'VCTK_cv057', 'VCTK_cv058', 'VCTK_cv059', 'VCTK_cv060', 'VCTK_cv061', 'VCTK_cv062', 'VCTK_cv063', 'VCTK_cv064', 'VCTK_cv065', 'VCTK_cv066', 'VCTK_cv067', 'VCTK_cv068', 'VCTK_cv070', 'VCTK_cv071', 'VCTK_cv072', 'VCTK_cv073', 'VCTK_cv074', 'VCTK_cv075', 'VCTK_cv076', 'VCTK_cv077', 'VCTK_c

/home/pruuwu/.cache/pypoetry/virtualenvs/khongkhuntts-t5CDf9ik-py3.11/lib/python3.11/site-packages/torch/utils/data/sampler.py:77: UserWarning: `data_source` argument is not used and will be removed in 2.2.0.You may still have custom implementation that utilizes it.
  warnings.warn(
/home/pruuwu/.cache/pypoetry/virtualenvs/khongkhuntts-t5CDf9ik-py3.11/lib/python3.11/site-packages/TTS/tts/models/vits.py:1273: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):  # use float32 for the criterion
/home/pruuwu/.cache/pypoetry/virtualenvs/khongkhuntts-t5CDf9ik-py3.11/lib/python3.11/site-packages/TTS/tts/models/vits.py:1284: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/home/pruuwu/.cache/pypoetry/virtualenvs/khongkhuntts-t5CDf9ik-py3.11/lib/python3.11/site-packages/TTS/tts/models/vit

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.0527120033899943 (+0.005519747734069824)
     | > avg_loss_disc: 2.2594182888666787 (-0.001241286595662583)
     | > avg_loss_disc_real_0: 0.13096010064085326 (+0.005591159065564483)
     | > avg_loss_disc_real_1: 0.20639177163441977 (+0.0017867932717005597)
     | > avg_loss_disc_real_2: 0.25274282942215603 (+0.04661465436220172)
     | > avg_loss_disc_real_3: 0.20845249791940054 (-0.007377311587333679)
     | > avg_loss_disc_real_4: 0.24129488319158554 (+0.020830996334552765)
     | > avg_loss_disc_real_5: 0.23739424844582876 (+0.012924114863077818)
     | > avg_loss_0: 2.2594182888666787 (-0.001241286595662583)
     | > avg_loss_spk_encoder: -6.489972909291585 (-0.06318092346191406)
     | > avg_loss_gen: 2.822462717692057 (+0.13558447360992432)
     | > avg_loss_kl: 3.5503104527791343 (+0.1652451753616333)
     | > avg_loss_feat: 7.453443209330241 (+0.07025440533955951)
     | > avg_loss_mel: 19.21384334564209 (-0.2318436304728202

it'` s the tip of the iceberg.
 [!] Character '`' not found in the vocabulary. Discarding it.



   --> TIME: 2025-04-30 21:30:49 -- STEP: 1360/4163 -- GLOBAL_STEP: 290500
     | > loss_disc: 2.1481292247772217  (2.132110191180426)
     | > loss_disc_real_0: 0.09249359369277954  (0.10606613835210309)
     | > loss_disc_real_1: 0.20606762170791626  (0.1837763060267795)
     | > loss_disc_real_2: 0.1670520156621933  (0.19472980909597348)
     | > loss_disc_real_3: 0.19226333498954773  (0.20151457989259677)
     | > loss_disc_real_4: 0.2309064269065857  (0.2007934583022314)
     | > loss_disc_real_5: 0.22127074003219604  (0.20880239896257138)
     | > loss_0: 2.1481292247772217  (2.132110191180426)
     | > grad_norm_0: tensor(6.3340, device='cuda:0')  (tensor(6.6011, device='cuda:0'))
     | > loss_spk_encoder: -6.558933258056641  (-6.43991810223636)
     | > loss_gen: 2.887770652770996  (2.8326732346240204)
     | > loss_kl: 3.486823797225952  (3.4098895313108657)
     | > loss_feat: 7.948178291320801  (7.8988730237764475)
     | > loss_mel: 18.9992618560791  (19.914477676503772)


 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.04643197854359945 (-0.006280024846394852)
     | > avg_loss_disc: 2.200575828552246 (-0.05884246031443263)
     | > avg_loss_disc_real_0: 0.0923897127310435 (-0.03857038790980975)
     | > avg_loss_disc_real_1: 0.18045153468847275 (-0.02594023694594702)
     | > avg_loss_disc_real_2: 0.1809858779112498 (-0.07175695151090625)
     | > avg_loss_disc_real_3: 0.20202779521544775 (-0.006424702703952789)
     | > avg_loss_disc_real_4: 0.21908149868249893 (-0.02221338450908661)
     | > avg_loss_disc_real_5: 0.21401762713988623 (-0.023376621305942535)
     | > avg_loss_0: 2.200575828552246 (-0.05884246031443263)
     | > avg_loss_spk_encoder: -6.47999922434489 (+0.009973684946695371)
     | > avg_loss_gen: 2.607436021169027 (-0.2150266965230303)
     | > avg_loss_kl: 3.5242493549982705 (-0.02606109778086374)
     | > avg_loss_feat: 7.78005329767863 (+0.32661008834838867)
     | > avg_loss_mel: 19.28219000498454 (+0.0683466593424491)
     | >

In [ ]:
trainer.save_checkpoint()